# 07 — ASML-Style Wafer Metrology Capstone

## Objective

Combine the framework's full metrology chain in one high-end semiconductor workflow, inspired by wafer inspection in ASML-style lithography systems: wafer pattern, engineered surface roughness, coherent speckle noise, defect detection, and signal-to-noise analysis.

## What you'll see

- A wafer-like surface with roughness and defects under coherent illumination (speckle)
- Defect detection (`DefectAnalyzer`) and error-map comparison (`ErrorMapAnalyzer`)
- Signal-to-noise and speckle-roughness metrics (`SNRAnalyzer`, `SpeckleRoughnessEstimator`)

## How to use this notebook

1. Edit the parameters in the next code cell to change wafer roughness, defect depth, and exposure.
2. Run the simulation cells in order to generate the synthetic image and metrics.
3. Use the summary values to compare defect detectability and SNR under different settings.
4. Try the optimization cell to explore how coherence and exposure influence metrology performance.

In [ ]:
# Editable parameters for the capstone workflow
size = 96
spacing = 0.5
wavelength = 193e-9  # DUV-like wavelength
exposure_time = 1e-3
coherence_length = 1e-4
roughness_amplitude = 0.20
defect_depth = 2.0
defect_radius = 5

print('Configuration:')
print(f'  size={size}')
print(f'  wavelength={wavelength:.1e} m')
print(f'  exposure_time={exposure_time:.3e} s')
print(f'  coherence_length={coherence_length:.1e} m')
print(f'  roughness_amplitude={roughness_amplitude}')
print(f'  defect_depth={defect_depth}')

In [ ]:
import numpy as np

from optical_metrology.analysis import (
    DefectAnalyzer,
    ErrorMapAnalyzer,
    SNRAnalyzer,
    SpeckleRoughnessEstimator,
)
from optical_metrology.detector import CMOSDetector
from optical_metrology.detector.noise_models import SpeckleNoise
from optical_metrology.illumination import bright_field
from optical_metrology.optics import GaussianPSF, OpticalPropagator, OpticalSystem
from optical_metrology.scattering import LambertianScattering
from optical_metrology.surface import RoughSurface, WaferSurface
from optical_metrology.surface.base import GeometryAnalyzer, Material

In [ ]:
def create_capstone_surface(
    shape,
    roughness_amplitude,
    defect_depth,
    defect_radius,
    defect_center=None,
):
    wafer = WaferSurface(
        shape=shape,
        die_rows=4,
        die_cols=4,
        street_width=3,
        fiducial_size=4,
        fiducial_height=1.8,
        die_height_val=1.0,
    )
    rough = RoughSurface(shape=shape, sigma=8.0, amplitude=roughness_amplitude)

    height = wafer.height + 0.12 * rough.height
    if defect_center is None:
        defect_center = (shape[0] * 3 // 4, shape[1] // 2)

    yy, xx = np.mgrid[: shape[0], : shape[1]]
    distance_sq = (yy - defect_center[0]) ** 2 + (xx - defect_center[1]) ** 2
    height -= defect_depth * np.exp(-distance_sq / (2.0 * defect_radius ** 2))

    return GeometryAnalyzer.analyze(height, material=Material('silicon', refractive_index=3.9))

def capture_simulated_image(
    surface,
    wavelength,
    spacing,
    exposure_time,
    coherence_length,
    rng_seed,
):
    source = bright_field(wavelength=wavelength, power=1.0, incidence_angle=0.05)
    lf = source.generate_light_field(shape=surface.height.shape, spacing=spacing)

    scatter = LambertianScattering(albedo=0.75)
    scattered = scatter.evaluate(lf, surface, view_direction=np.array([0.0, 0.0, 1.0]))

    optics = OpticalSystem(
        wavelength=wavelength,
        aperture_diameter=8e-3,
        focal_length=50e-3,
        magnification=1.0,
    )
    propagator = OpticalPropagator(GaussianPSF(sigma=1.2), throughput_enabled=True)
    sensor = propagator.propagate(scattered, optics)

    detector = CMOSDetector(
        exposure_time=exposure_time,
        quantum_efficiency=0.45,
        read_noise_sigma=1.8,
        gain=2.0,
        bit_depth=12,
        pixel_area=25e-12,
        rng_seed=rng_seed,
        noise_models=[SpeckleNoise(coherence_length=coherence_length)],
    )
    return detector.capture(sensor, surface=surface)

def capture_reference_image(surface, wavelength, spacing, exposure_time):
    source = bright_field(wavelength=wavelength, power=1.0, incidence_angle=0.05)
    lf = source.generate_light_field(shape=surface.height.shape, spacing=spacing)
    scatter = LambertianScattering(albedo=0.75)
    scattered = scatter.evaluate(lf, surface, view_direction=np.array([0.0, 0.0, 1.0]))
    optics = OpticalSystem(
        wavelength=wavelength,
        aperture_diameter=8e-3,
        focal_length=50e-3,
        magnification=1.0,
    )
    propagator = OpticalPropagator(GaussianPSF(sigma=1.2), throughput_enabled=True)
    sensor = propagator.propagate(scattered, optics)
    detector = CMOSDetector(
        exposure_time=exposure_time,
        quantum_efficiency=0.45,
        read_noise_sigma=0.0,
        gain=2.0,
        bit_depth=12,
        pixel_area=25e-12,
        rng_seed=0,
        noise_models=[],
    )
    return detector.capture(sensor, surface=surface)

In [ ]:
reference_surface = create_capstone_surface(
    shape=(size, size),
    roughness_amplitude=roughness_amplitude,
    defect_depth=0.0,
    defect_radius=defect_radius,
)
defect_surface = create_capstone_surface(
    shape=(size, size),
    roughness_amplitude=roughness_amplitude,
    defect_depth=defect_depth,
    defect_radius=defect_radius,
)

reference_image = capture_reference_image(reference_surface, wavelength, spacing, exposure_time)
defect_image = capture_simulated_image(defect_surface, wavelength, spacing, exposure_time, coherence_length, rng_seed=5)

print('Reference image')
print(reference_image.visualize(max_width=64))
print('Defect image')
print(defect_image.visualize(max_width=64))

In [ ]:
defect_analyzer = DefectAnalyzer(threshold=0.06, min_area=8, max_area=800, reference_image=reference_image.pixels)
defect_report = defect_analyzer.analyze(defect_image)

error_analyzer = ErrorMapAnalyzer(reference_image)
error_report = error_analyzer.analyze(defect_image)

snr_analyzer = SNRAnalyzer(
    method='single_image',
    signal_region=(20, 20, 40, 40),
    noise_region=(0, 0, 12, 12),
)
snr_report = snr_analyzer.analyze(defect_image)

roughness_analyzer = SpeckleRoughnessEstimator(
    coherence_length=coherence_length,
    wavelength=wavelength,
    roi=(20, 20, 40, 40),
)
roughness_report = roughness_analyzer.analyze(defect_image)

print('ASML-style defect inspection results')
print(f'  defect_count={defect_report.measurements["defect_count"]}')
print(f'  total_defect_area={defect_report.measurements["total_defect_area"]:.1f} pixels')
print(f'  snr_db={snr_report.measurements["snr_db"]:.2f} dB')
print(f'  speckle_contrast={roughness_report.measurements["speckle_contrast"]:.3f}')
print(f'  estimated_roughness_rms={roughness_report.measurements["estimated_roughness_rms"]:.4f} m')
print(f'  error_rmse={error_report.measurements["rmse"]:.4f}')
print(f'  error_psnr={error_report.measurements["psnr_db"]:.2f} dB')

for defect in defect_report.measurements["defects"]:
    print(f'  - label={defect["label"]} type={defect["defect_type"]} area={defect["area"]} bbox={defect["bbox"]}')

In [ ]:
exposure_options = [0.5e-3, 1.0e-3, 2.0e-3]
exposure_results = []
for exposure in exposure_options:
    image = capture_simulated_image(defect_surface, wavelength, spacing, exposure, coherence_length, rng_seed=5)
    defect_report = DefectAnalyzer(threshold=0.06, min_area=8, max_area=800, reference_image=reference_image.pixels).analyze(image)
    snr_report = SNRAnalyzer(
        method='single_image',
        signal_region=(20, 20, 40, 40),
        noise_region=(0, 0, 12, 12),
    ).analyze(image)
    exposure_results.append((exposure, defect_report.measurements['defect_count'], snr_report.measurements['snr_db']))

print('Exposure scan: defect count and SNR')
for exposure, defect_count, snr_db in exposure_results:
    print(f'  exposure={exposure:.3e} s  defect_count={defect_count}  snr_db={snr_db:.2f} dB')

In [ ]:
coherence_options = [2e-5, 1e-4, 5e-4]
coherence_results = []
for coherence in coherence_options:
    image = capture_simulated_image(defect_surface, wavelength, spacing, exposure_time, coherence, rng_seed=5)
    snr_report = SNRAnalyzer(
        method='single_image',
        signal_region=(20, 20, 40, 40),
        noise_region=(0, 0, 12, 12),
    ).analyze(image)
    roughness_report = SpeckleRoughnessEstimator(
        coherence_length=coherence,
        wavelength=wavelength,
        roi=(20, 20, 40, 40),
    ).analyze(image)
    coherence_results.append((coherence, snr_report.measurements['snr_db'], roughness_report.measurements['speckle_contrast']))

print('Coherence scan: SNR and speckle contrast')
for coherence, snr_db, speckle_contrast in coherence_results:
    print(f'  coherence={coherence:.1e} m  snr_db={snr_db:.2f} dB  speckle_contrast={speckle_contrast:.3f}')

In [ ]:
defect_depth_options = [0.5, 1.0, 2.0, 3.0]
defect_depth_results = []
for depth in defect_depth_options:
    defect_surface = create_capstone_surface(
        shape=(size, size),
        roughness_amplitude=roughness_amplitude,
        defect_depth=depth,
        defect_radius=defect_radius,
    )
    defect_image = capture_simulated_image(defect_surface, wavelength, spacing, exposure_time, coherence_length, rng_seed=5)
    defect_report = DefectAnalyzer(threshold=0.06, min_area=8, max_area=800, reference_image=reference_image.pixels).analyze(defect_image)
    defect_depth_results.append((depth, defect_report.measurements['defect_count'], defect_report.measurements['total_defect_area']))

print('Defect depth scan: detectability')
for depth, defect_count, total_area in defect_depth_results:
    print(f'  defect_depth={depth:.1f}  defect_count={defect_count}  total_area={total_area:.1f} px')

In [ ]:
flat_reference_surface = create_capstone_surface(
    shape=(size, size),
    roughness_amplitude=roughness_amplitude,
    defect_depth=0.0,
    defect_radius=defect_radius,
)
flat_reference_image = capture_reference_image(flat_reference_surface, wavelength, spacing, exposure_time)
flat_ref_report = DefectAnalyzer(
    threshold=0.06,
    min_area=8,
    max_area=800,
    reference_image=flat_reference_image.pixels,
).analyze(defect_image)
no_ref_report = DefectAnalyzer(
    threshold=0.06,
    min_area=8,
    max_area=800,
    reference_image=None,
).analyze(defect_image)

print('Reference strategy comparison:')
print('  using flat-field reference:')
print(f'    defect_count={flat_ref_report.measurements["defect_count"]}')
print(f'    total_area={flat_ref_report.measurements["total_defect_area"]:.1f} px')
print('  using threshold-only detection:')
print(f'    defect_count={no_ref_report.measurements["defect_count"]}')
print(f'    total_area={no_ref_report.measurements["total_defect_area"]:.1f} px')

## Final summary and inline plots

The following table summarizes the UC7 capstone metrics, and the plots visualize the parameter sweeps for exposure, coherence, defect depth, and reference strategy.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

summary_table = [
    ["Metric", "Value"],
    ["defect_count", defect_report.measurements["defect_count"]],
    ["total_defect_area", f"{defect_report.measurements['total_defect_area']:.1f} px"],
    ["snr_db", f"{snr_report.measurements['snr_db']:.2f}"],
    ["speckle_contrast", f"{roughness_report.measurements['speckle_contrast']:.3f}"],
    ["estimated_roughness_rms", f"{roughness_report.measurements['estimated_roughness_rms']:.4f} m"],
    ["error_rmse", f"{error_report.measurements['rmse']:.4f}"],
    ["error_psnr", f"{error_report.measurements['psnr_db']:.2f} dB"],
]

fig, ax = plt.subplots(figsize=(6, 2))
ax.axis('off')
ax.table(cellText=summary_table, loc='center', cellLoc='left')
ax.set_title('UC7 Capstone Inspection Summary', pad=12)
plt.show()

exposure_arr = np.array(exposure_results, dtype=float)
fig, ax1 = plt.subplots(figsize=(7, 3))
ax1.plot(exposure_arr[:, 0] * 1e3, exposure_arr[:, 2], marker='o', label='SNR (dB)')
ax1.set_xlabel('Exposure time (ms)')
ax1.set_ylabel('SNR (dB)')
ax2 = ax1.twinx()
ax2.plot(exposure_arr[:, 0] * 1e3, exposure_arr[:, 1], marker='s', color='tab:orange', label='Defect count')
ax2.set_ylabel('Defect count')
ax1.set_title('Exposure scan')
ax1.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

coherence_arr = np.array(coherence_results, dtype=float)
fig, ax1 = plt.subplots(figsize=(7, 3))
ax1.plot(coherence_arr[:, 0] * 1e6, coherence_arr[:, 1], marker='o', label='SNR (dB)')
ax1.set_xlabel('Coherence length (µm)')
ax1.set_ylabel('SNR (dB)')
ax2 = ax1.twinx()
ax2.plot(coherence_arr[:, 0] * 1e6, coherence_arr[:, 2], marker='s', color='tab:orange', label='Speckle contrast')
ax2.set_ylabel('Speckle contrast')
ax1.set_title('Coherence scan')
ax1.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

defect_depth_arr = np.array(defect_depth_results, dtype=float)
fig, ax1 = plt.subplots(figsize=(7, 3))
ax1.plot(defect_depth_arr[:, 0], defect_depth_arr[:, 1], marker='o', label='Defect count')
ax1.set_xlabel('Defect depth')
ax1.set_ylabel('Defect count')
ax2 = ax1.twinx()
ax2.plot(defect_depth_arr[:, 0], defect_depth_arr[:, 2], marker='s', color='tab:orange', label='Total defect area')
ax2.set_ylabel('Total defect area (px)')
ax1.set_title('Defect depth scan')
ax1.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

strategy_methods = ['flat_ref', 'threshold_only']
strategy_counts = [
    flat_ref_report.measurements['defect_count'],
    no_ref_report.measurements['defect_count'],
]
strategy_areas = [
    flat_ref_report.measurements['total_defect_area'],
    no_ref_report.measurements['total_defect_area'],
]
fig, ax = plt.subplots(figsize=(7, 3))
width = 0.35
x = np.arange(len(strategy_methods))
ax.bar(x - width/2, strategy_counts, width, label='Defect count')
ax.bar(x + width/2, strategy_areas, width, label='Total defect area (px)')
ax.set_xticks(x)
ax.set_xticklabels(strategy_methods)
ax.set_title('Reference strategy comparison')
ax.legend()
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
plt.show()

## Try next

- Change the exposure time and observe how SNR and defect count respond in the exposure scan cell.
- Reduce the coherence length in the coherence scan cell to see how speckle contrast and SNR trade off.
- Adjust the defect depth in the defect depth scan cell to explore the threshold for reliable detection.
- Replace the reference image with a flat-field capture by setting the reference surface to a rough wafer without defects, then re-run the defect analyzer. This compares true reference-based detection versus intensity-only thresholding.